# Regression – Preise vorhersagen mit LEGO

In diesem Notebook lernen wir eine der wichtigsten Techniken im Machine Learning kennen: die **Regression**.
Als Datensatz nutzen wir echte LEGO-Sets – mit Preisen, Bausteinen, Themen und mehr!

## Was ist Regression?

Beim Machine Learning gibt es zwei große Aufgaben:

1. **Klassifikation** – Dinge in Kategorien einteilen. Zum Beispiel: *Ist diese E-Mail Spam oder nicht?*
2. **Regression** – Eine **Zahl** vorhersagen. Zum Beispiel: *Was kostet dieses LEGO-Set?*

Heute geht es um die **Regression**. Wir wollen herausfinden:

*Kann ein Computer den Preis eines LEGO-Sets vorhersagen – nur anhand von Infos wie der Anzahl der Teile?*

Aber bevor wir einen Computer trainieren können, müssen wir die Daten **verstehen**. Das heißt: Erst schauen, dann rechnen!

---

## Teil 1: Daten laden mit Pandas

Unser Datensatz stammt von **Kaggle** – einer Plattform für Data Scientists. Er enthält hunderte von LEGO-Sets mit ihrem offiziellen Preis, der Anzahl der Teile, dem Thema und mehr. Die Originaldaten findest du unter [kaggle.com/datasets/mterzolo/lego-sets](https://www.kaggle.com/datasets/mterzolo/lego-sets).

**Pandas** ist die wichtigste Bibliothek, um Tabellen in Python zu verarbeiten.
Wir laden die CSV-Datei in einen sogenannten **DataFrame** – das ist wie eine Excel-Tabelle direkt im Notebook.

In [ ]:
import pandas as pd

# Datei einlesen
df = pd.read_csv("data/lego_sets.csv")

print(f"Datensatz geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")

---

## Teil 2: Erste Erkundung – Was steckt im Datensatz?

Bevor wir irgendetwas mit Machine Learning machen, schauen wir uns die Daten ganz genau an.
Data Scientists nennen das **Exploratory Data Analysis (EDA)**.

### 2.1 – Die ersten Zeilen

Mit `.head()` zeigen wir die ersten 5 Zeilen der Tabelle an. So bekommen wir sofort ein Gefühl für die Daten.

In [ ]:
df.head()

### 2.2 – Welche Spalten gibt es?

Mit `.info()` sehen wir alle Spalten, ihren Datentyp und wie viele Werte vorhanden sind.
Fehlende Werte (`NaN`) können später ein Problem sein – gut, das früh zu wissen!

In [ ]:
df.info()

### 2.3 – Statistische Kennzahlen

Mit `.describe()` bekommen wir für jede **Zahlenspalte** automatisch wichtige Kennzahlen:

| Kennzahl | Bedeutung |
|---|---|
| `count` | Wie viele Werte vorhanden sind |
| `mean` | Der Durchschnitt |
| `min` / `max` | Der kleinste / größte Wert |
| `50%` | Der Median (die Mitte) |

In [ ]:
df.describe()

### 2.4 – Duplikate: Warum tauchen Sets mehrfach auf?

Im Datensatz gibt es eine Spalte `country`. Viele Sets sind für **mehrere Länder** erfasst – mit leicht unterschiedlichen Preisen. Das bedeutet: Ein und dasselbe Set taucht mehrfach auf!

Schauen wir uns das am Beispiel des **Millennium Falcon** an (7.541 Teile, `piece_count = 7541`):

In [ ]:
# Den großen Millennium Falcon (7541 Teile) im Datensatz suchen
falcon_alle = df[(df["set_name"].str.contains("Millennium", case=False, na=False)) & (df["piece_count"] == 7541)]
falcon_alle[["set_name", "piece_count", "list_price", "country"]]

Dasselbe Set erscheint für viele Länder mit unterschiedlichen Preisen! Damit unser Modell nicht durch Duplikate verfälscht wird, filtern wir auf ein einzelnes Land.

Wir nehmen **US**, weil die Preise in US-Dollar angegeben sind und die meisten Sets abgedeckt sind:

In [ ]:
print(f"Vorher:  {len(df)} Zeilen ({df['country'].nunique()} Länder: {', '.join(df['country'].unique())})")

df = df[df["country"] == "US"].copy()

print(f"Nachher: {len(df)} Zeilen (nur US)")

---

## Teil 3: Daten visualisieren

Zahlen lesen ist gut – aber **Bilder** helfen uns, Muster viel schneller zu erkennen.
Wir nutzen **Matplotlib**, die Standard-Bibliothek für Diagramme in Python.

### 3.1 – Wie teuer sind LEGO-Sets? (Histogramm)

Ein **Histogramm** zeigt, wie häufig bestimmte Werte vorkommen.
Wir schauen uns an: Wie sind die Preise verteilt?

In [ ]:
import matplotlib.pyplot as plt

preise = df["list_price"].dropna()

preise.plot.hist(bins=40, color="steelblue", edgecolor="white", figsize=(10, 5))
plt.title("Verteilung der LEGO-Preise")
plt.xlabel("Preis (USD)")
plt.ylabel("Anzahl Sets")
plt.tight_layout()
plt.show()

print(f"Günstigstes Set:    {preise.min():.2f}")
print(f"Teuerstes Set:      {preise.max():.2f}")
print(f"Durchschnittspreis: {preise.mean():.2f}")

### 3.2 – Mehr Teile = Höherer Preis? (Streudiagramm)

Jetzt kommt die entscheidende Frage: Gibt es einen **Zusammenhang** zwischen der Anzahl der Teile und dem Preis?

Ein **Streudiagramm** (Scatter Plot) zeigt jeden Datenpunkt als einen Punkt. So sehen wir sofort, ob es einen Trend gibt.

In [ ]:
plot_df = df[["piece_count", "list_price"]].dropna()

plot_df.plot.scatter(x="piece_count", y="list_price",
                     alpha=0.4, color="darkorange", edgecolors="none", s=20,
                     figsize=(10, 6))
plt.title("Anzahl der Teile vs. Preis")
plt.xlabel("Anzahl Teile")
plt.ylabel("Preis (USD)")
plt.tight_layout()
plt.show()
print("Was fällt dir auf? Gibt es einen Trend?")

### 3.3 – Die beliebtesten Themen

LEGO-Sets gehören zu verschiedenen **Themen** (Themes) wie City, Technic, Star Wars usw.
Welche Themen kommen am häufigsten vor?

In [ ]:
top_themen = df["theme_name"].value_counts().head(10)

top_themen.plot.barh(color="mediumseagreen", edgecolor="white", figsize=(10, 5))
plt.title("Top 10 LEGO-Themen im Datensatz")
plt.xlabel("Anzahl Sets")
plt.gca().invert_yaxis()  # Größte Zahl oben
plt.tight_layout()
plt.show()

---

## Teil 4: Fehlende Werte

In echten Datensätzen fehlen oft Werte. Das nennt man **Missing Values** (`NaN` = Not a Number).
Bevor wir einen Algorithmus trainieren, müssen wir damit umgehen.

**Wie viele Werte fehlen pro Spalte?**

In [ ]:
fehlende_werte = df.isnull().sum()
fehlende_werte = fehlende_werte[fehlende_werte > 0].sort_values(ascending=False)

for spalte, anzahl in fehlende_werte.items():
    prozent = anzahl / len(df) * 100
    print(f"   {spalte:<30} {anzahl:>5} Werte fehlen ({prozent:.1f}%)")

---

## Zusammenfassung & Ausblick

Das haben wir zunächst gemacht:

*   Die Daten mit Pandas geladen und untersucht (`.head()`, `.info()`, `.describe()`)
*   Die Preisverteilung als **Histogramm** visualisiert
*   Den Zusammenhang Teile vs. Preis als **Streudiagramm** untersucht
*   Die beliebtesten LEGO-Themen als **Balkendiagramm** angezeigt
*   **Fehlende Werte** identifiziert

### Was kommt als nächstes?

Im nächsten Schritt trainieren wir unser **erstes Regressionsmodell**.
Wir werden eine Regressionslinie durch das Streudiagramm legen und damit Preise vorhersagen.

Die entscheidende Frage bleibt: *Wie gut kann ein Modell den Preis eines LEGO-Sets vorhersagen – nur aus der Anzahl der Teile?*

---

# Übungsaufgaben – Regression mit LEGO-Daten

Jetzt bist du dran! In den folgenden Aufgaben trainierst du dein **erstes Regressionsmodell** und analysierst, wie gut es funktioniert.

Wir nutzen dafür **scikit-learn** – die wichtigste Python-Bibliothek für Machine Learning.

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

### Aufgabe 1: Lineare Regression – Teileanzahl vs. Preis

Jetzt trainieren wir unser erstes **lineares Regressionsmodell**! Die Idee dahinter ist einfach:

> Wir suchen eine **Gerade** (eine Linie), die möglichst gut durch die Punktwolke im Streudiagramm passt.

Die Formel einer Geraden kennst du vielleicht aus dem Mathe-Unterricht:

$$\text{Preis} = m \cdot \text{Teileanzahl} + b$$

- **m** = die Steigung (wie viel Euro kostet ein zusätzliches Teil im Durchschnitt?)
- **b** = der y-Achsenabschnitt (Grundpreis, wenn ein Set 0 Teile hätte)

**Schritt für Schritt:**

1. Bereite die Daten vor: Entferne alle Zeilen, in denen `piece_count` oder `list_price` fehlen (`.dropna()`).
2. Erstelle ein Modell mit `LinearRegression()` und trainiere es mit `.fit()`.
3. Plotte das **Streudiagramm** (Teile vs. Preis) und zeichne die **Regressionsgerade** darüber.
4. Gib **Steigung (m)** und **y-Achsenabschnitt (b)** aus.

*Tipp:* So trainierst du ein lineares Modell mit scikit-learn:
```python
modell = LinearRegression()
modell.fit(X, y)           # X = Eingabe (2D-Array), y = Zielwert (1D)
modell.coef_[0]            # Steigung m
modell.intercept_          # y-Achsenabschnitt b
modell.predict(X)          # Vorhersagen berechnen
```
**Wichtig:** `X` muss ein 2D-Array sein. Nutze `df[["spalte"]]` (doppelte Klammern!) statt `df["spalte"]`.

In [ ]:
# Aufgabe 1: Lineare Regression trainieren

# 1. Daten vorbereiten: nur Zeilen mit Teileanzahl UND Preis
reg_df = df[["piece_count", "list_price"]].dropna()

X = reg_df[["piece_count"]]   # 2D-Array (Eingabe)
y = reg_df["list_price"]       # 1D-Array (Zielwert)

# 2. Modell erstellen und trainieren
modell = LinearRegression()
# modell.fit(...)

m = modell.coef_[0]
b = modell.intercept_

# print(f"Steigung (m): ...")
# print(f"y-Achsenabschnitt (b): ...")

# 3. Streudiagramm + Regressionsgerade
# reg_df.plot.scatter(...)
# plt.plot(...)
# plt.show()


### Aufgabe 2: Schnäppchen und Preistreiber finden

Unser Modell sagt für jedes Set einen Preis vorher. Manche Sets sind aber **viel teurer** als vorhergesagt (Preistreiber), andere **viel günstiger** (Schnäppchen).

Der Unterschied zwischen dem echten Preis und der Vorhersage heißt **Residuum**:

$$\text{Residuum} = \text{echter Preis} - \text{vorhergesagter Preis}$$

- **Residuum > 0** → Das Set ist **teurer** als erwartet (Preistreiber)
- **Residuum < 0** → Das Set ist **günstiger** als erwartet (Schnäppchen)

**Aufgabe:**

1. Berechne für jedes Set im `reg_df` die **Vorhersage** des Modells und speichere sie in einer neuen Spalte `vorhersage`.
2. Berechne das **Residuum** (echter Preis minus Vorhersage) und speichere es in einer neuen Spalte `residuum`.
3. Du brauchst auch die Set-Namen! Erstelle dazu einen neuen DataFrame, der **alle Spalten** enthält (so wie in `df`), aber nur die Zeilen, bei denen `piece_count` und `list_price` nicht fehlen. Füge dort die Spalten `vorhersage` und `residuum` hinzu.
4. Zeige die **5 teuersten Sets** relativ zur Vorhersage (höchstes Residuum).
5. Zeige die **5 günstigsten Sets** relativ zur Vorhersage (niedrigstes Residuum).

*Tipp:* Nutze `.nlargest(5, "residuum")` und `.nsmallest(5, "residuum")` um die Extremwerte zu finden. Zeige die Spalten `set_name`, `piece_count`, `list_price`, `vorhersage`, `residuum`.

In [ ]:
# Aufgabe 2: Schnäppchen und Preistreiber finden

# DataFrame mit allen Spalten, aber nur vollständige Zeilen
analyse_df = df.dropna(subset=["piece_count", "list_price"]).copy()

# Vorhersage und Residuum berechnen
# analyse_df["vorhersage"] = modell.predict(...)
# analyse_df["residuum"] = ...

anzeige = ["set_name", "theme_name", "piece_count", "list_price", "vorhersage", "residuum"]

# Top 10 Preistreiber
print("Top 10 PREISTREIBER (viel teurer als vorhergesagt):\n")
# print(analyse_df.nlargest(...)[anzeige].to_string(index=False))

# Top 10 Schnäppchen
print("Top 10 SCHNÄPPCHEN (viel günstiger als vorhergesagt):\n")
# print(analyse_df.nsmallest(...)[anzeige].to_string(index=False))


### Aufgabe 3: Mittlere relative Abweichung pro Themenwelt

Unser Modell funktioniert nicht für alle Themen gleich gut. Ein **Star-Wars**-Set hat vielleicht einen Lizenzaufschlag, während ein einfaches **City**-Set nah an der Vorhersage liegt.

Um das zu messen, berechnen wir die **mittlere relative Abweichung** pro Thema – diesmal **mit Vorzeichen**:

$$\text{Relative Abweichung} = \frac{\text{echter Preis} - \text{vorhergesagter Preis}}{\text{vorhergesagter Preis}} \times 100\%$$

- **Positiv** → Sets dieses Themas sind im Durchschnitt **teurer** als vorhergesagt (z.B. Lizenzaufschlag)
- **Negativ** → Sets dieses Themas sind im Durchschnitt **günstiger** als vorhergesagt

**Aufgabe:**

1. Berechne für jedes Set die **vorzeichenbehaftete relative Abweichung in Prozent** und speichere sie in einer neuen Spalte `abweichung_prozent`.
2. Finde die **10 beliebtesten Themen** (die mit den meisten Sets im Datensatz).
3. Filtere auf diese 10 Themen und berechne die **mittlere relative Abweichung** pro Thema.
4. Erstelle ein **horizontales Balkendiagramm**, das die mittlere Abweichung dieser 10 Themen zeigt – sortiert nach Abweichung. Positive Balken sollten sich farblich von negativen unterscheiden.

*Tipps:*
- Diesmal **ohne** `abs()`: `(analyse_df["residuum"] / analyse_df["vorhersage"]) * 100`
- Die 10 beliebtesten Themen findest du mit `value_counts().head(10).index`
- Für unterschiedliche Farben: `["seagreen" if v < 0 else "coral" for v in werte]`

In [ ]:
# Aufgabe 3: Mittlere relative Abweichung pro Themenwelt

# 1. Vorzeichenbehaftete relative Abweichung in Prozent
# analyse_df["abweichung_prozent"] = (analyse_df["residuum"] / ...) * 100

# 2. Die 10 beliebtesten Themen (meiste Sets)
top10_themen = analyse_df["theme_name"].value_counts().head(10).index

# 3. Nur diese 10 Themen → mittlere Abweichung pro Thema
# analyse_top = analyse_df[...]
# mape_pro_thema = analyse_top.groupby("theme_name")["abweichung_prozent"].mean().sort_values()

# 4. Balkendiagramm mit Farben nach Vorzeichen
# farben = ["seagreen" if v < 0 else "coral" for v in mape_pro_thema]
# mape_pro_thema.plot.barh(...)
# plt.show()


### Aufgabe 4: Ist der Millennium Falcon sein Geld wert?

Der **LEGO Millennium Falcon** (Set 75192) ist eines der größten und teuersten LEGO-Sets aller Zeiten – mit über 7.500 Teilen und einem Preis von knapp 800 $.

Aber ist der Preis **gerechtfertigt**, wenn wir unser Regressionsmodell fragen? Oder zahlt man hier einen kräftigen **Star-Wars-Aufschlag**?

**Aufgabe:**

1. Finde den Millennium Falcon im Datensatz. *Tipp:* Suche in der Spalte `set_name` nach `"Millennium"` mit `.str.contains()`.
2. Zeige seine wichtigsten Daten: `set_name`, `piece_count`, `list_price`, `theme_name`.
3. Nutze das Modell, um den **vorhergesagten Preis** basierend auf der Teileanzahl zu berechnen.
4. Berechne die **Abweichung** (echter Preis minus Vorhersage) und die **relative Abweichung in Prozent**.
5. Interpretiere: Ist der Millennium Falcon ein Schnäppchen oder ein Preistreiber?

In [ ]:
# Aufgabe 4: Ist der Millennium Falcon sein Geld wert?

# 1. Millennium Falcon im Datensatz finden
falcon = analyse_df[analyse_df["set_name"].str.contains("Millennium", case=False, na=False)]

# 2. Wichtigste Daten anzeigen
print("Alle Millennium-Falcon-Sets im Datensatz:\n")
# print(falcon[["set_name", "piece_count", "list_price", "theme_name"]].to_string(index=False))

# 3. Das teuerste Set herausgreifen
gross = falcon.nlargest(1, "list_price").iloc[0]

# 4. Vorhersage und Abweichung berechnen
# vorhersage = modell.predict([[gross["piece_count"]]])[0]
# abweichung = ...
# abweichung_pct = ...

# print(f"Echter Preis:         ...")
# print(f"Vorhergesagter Preis: ...")
# print(f"Abweichung:           ...")


### Profi-Aufgabe: Dein eigenes LEGO-Set bewerten

Stell dir vor, du designst ein eigenes LEGO-Set und willst wissen: **Was wäre ein fairer Preis?**

Nutze dein trainiertes Modell, um den Preis für ein Set mit einer beliebigen Teileanzahl vorherzusagen.

**Aufgabe:**

1. Wähle **3 verschiedene Teileanzahlen** (z.B. 100, 500, 2000).
2. Nutze `modell.predict(...)`, um die vorhergesagten Preise zu berechnen.
3. Gib die Ergebnisse schön formatiert aus.
4. **Bonusfrage:** Suche dir im Datensatz ein echtes Set mit einer ähnlichen Teileanzahl heraus. Wie nah liegt die Vorhersage am echten Preis?

*Tipp:* So sagst du einen Preis vorher:
```python
modell.predict([[500]])  # Vorhersage für ein Set mit 500 Teilen
```

In [ ]:
# Profi-Aufgabe: Dein eigenes LEGO-Set bewerten

# 1. Wähle 3 verschiedene Teileanzahlen
eigene_teile = [100, 500, 2000]

print("Preisvorhersagen für eigene Sets:\n")
for teile in eigene_teile:
    # preis = modell.predict([[teile]])[0]
    # print(f"   {teile:>5} Teile → vorhergesagter Preis: ...")
    pass

# 2. Vergleich mit echten Sets
print("\n--- Vergleich mit echten Sets ---\n")
# Tipp: Finde das nächste Set mit (analyse_df["piece_count"] - teile).abs()


### Aufgabe 5: Wie gut ist unser Modell? – Das Bestimmtheitsmaß R²

Wir haben unser Modell trainiert und Vorhersagen gemacht – aber **wie gut** ist es eigentlich insgesamt?

Dafür gibt es eine Kennzahl: das **Bestimmtheitsmaß** (auch **R²** genannt, gesprochen „R-Quadrat").

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

- $y_i$ = der echte Preis eines Sets
- $\hat{y}_i$ = der vom Modell vorhergesagte Preis
- $\bar{y}$ = der Durchschnittspreis aller Sets

**Was bedeutet R²?**

| R²-Wert | Bedeutung |
|---------|-----------|
| 1.0     | Perfekte Vorhersage – das Modell erklärt 100 % der Preisunterschiede |
| 0.8     | Sehr gut – 80 % der Unterschiede werden erklärt |
| 0.5     | Mittelmäßig – nur die Hälfte wird erklärt |
| 0.0     | Das Modell ist nicht besser als einfach den Durchschnitt zu raten |
| < 0     | Das Modell ist *schlechter* als der Durchschnitt (kommt selten vor) |

**Aufgabe:**

1. Berechne R² **von Hand**: Nutze die echten Preise (`y`), die Vorhersagen und den Mittelwert (`y.mean()`).
2. Vergleiche dein Ergebnis mit der eingebauten Methode `modell.score(X, y)`.
3. Interpretiere: Wie viel Prozent der Preisunterschiede erklärt die Teileanzahl allein?

In [ ]:
# Aufgabe 5: R² berechnen

# 1. R² von Hand berechnen
y_vorhersage = modell.predict(X).flatten()
y_mittelwert = y.mean()

# ss_res = ((y - y_vorhersage) ** 2).sum()   # Summe der quadrierten Residuen
# ss_tot = ((y - y_mittelwert) ** 2).sum()   # Gesamtstreuung
# r2_hand = 1 - ss_res / ss_tot

# print(f"R² (von Hand berechnet): {r2_hand:.4f}")

# 2. Vergleich mit der eingebauten Methode
# r2_sklearn = modell.score(X, y)
# print(f"R² (modell.score):       {r2_sklearn:.4f}")

# 3. Interpretation: Wie viel % der Preisunterschiede erklärt die Teileanzahl?


---

## Geschafft!

Du hast dein **erstes Regressionsmodell** trainiert und gelernt:

- Wie man mit **scikit-learn** eine lineare Regression erstellt
- Wie man eine **Regressionsgerade** in ein Streudiagramm zeichnet
- Was **Residuen** sind und wie man damit Ausreißer findet
- Wie man die **Vorhersagequalität pro Themenwelt** vergleicht (MAPE)
- Wie man mit dem **Bestimmtheitsmaß R²** die Güte eines Modells bewertet

Die wichtigste Erkenntnis: Ein einfaches Modell mit nur **einer** Eingabevariable (Teileanzahl) kann den Preis schon erstaunlich gut vorhersagen – aber es gibt Themen, bei denen es deutlich daneben liegt. Das ist der Startpunkt für komplexere Modelle!